# Taller 02 - Clasificación Supervisada con **Telco Customer Churn**
**Curso:** Inteligencia Artificial ECA&I - Posgrado  
**Universidad:** EAFIT  
**Problema:** Clasificación binaria (Churn: Yes/No)

Este notebook desarrolla el flujo completo solicitado para el problema de **clasificación** usando el dataset **Telco Customer Churn**:

1. **Análisis Exploratorio (EDA)**: correlaciones, valores faltantes y sesgos.  
2. **Preprocesamiento**: limpieza, codificación y escalamiento.  
3. **Ingeniería de Características**: selección de variables mediante métodos estadísticos.  
4. **Entrenamiento y Ajuste**: búsqueda de hiperparámetros con `GridSearchCV`.  
5. **Validación Robusta**: uso de **Cross-Validation** y reporte de métricas en **Test set**.  

Además, se comparan **3 modelos**:
- Logistic Regression
- K-Nearest Neighbors (KNN)
- Random Forest

## 0. Librerías y configuración

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 1. Carga del dataset

Se usa una copia pública del dataset **Telco Customer Churn**.  
Si prefieres trabajar con un archivo local, reemplaza la URL por la ruta de tu CSV.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print(f"Dimensiones iniciales: {df.shape}")
df.head()

### Conclusión breve
El dataset contiene información demográfica, contractual y de consumo de clientes de telecomunicaciones. La variable objetivo es `Churn`.

## 2. Análisis Exploratorio de Datos (EDA)

En esta sección se documentan:
- estructura general del dataset,
- valores faltantes,
- sesgo/desbalance de clases,
- relaciones entre variables,
- correlaciones entre variables numéricas.

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

### 2.1 Revisión de valores faltantes
En este dataset, `TotalCharges` suele llegar como texto y puede contener cadenas vacías que deben tratarse como faltantes.

In [ ]:
# Copia para EDA
eda_df = df.copy()

# Convertir TotalCharges a numérico, forzando errores a NaN
eda_df["TotalCharges"] = pd.to_numeric(eda_df["TotalCharges"], errors="coerce")

missing = eda_df.isna().sum().sort_values(ascending=False)
missing_pct = (eda_df.isna().mean() * 100).sort_values(ascending=False)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct.round(2)
})
missing_df[missing_df["missing_count"] > 0]

In [ ]:
plt.figure(figsize=(10, 4))
missing_df["missing_pct"].plot(kind="bar")
plt.title("Porcentaje de valores faltantes por variable")
plt.ylabel("% faltantes")
plt.xticks(rotation=45)
plt.show()

### 2.2 Identificación de sesgos / desbalance de clases
Se revisa la distribución de la variable objetivo para determinar si existe desbalance.

In [ ]:
target_dist = eda_df["Churn"].value_counts(dropna=False)
target_pct = eda_df["Churn"].value_counts(normalize=True).mul(100).round(2)

display(pd.DataFrame({
    "count": target_dist,
    "pct": target_pct
}))

plt.figure(figsize=(6, 4))
sns.countplot(data=eda_df, x="Churn")
plt.title("Distribución de la variable objetivo: Churn")
plt.show()

### Interpretación esperada
Si la clase `No` es claramente mayoritaria frente a `Yes`, existe un **sesgo de clase**.  
Esto hace que métricas como **accuracy** por sí solas sean insuficientes, por lo que también se reportarán **precision**, **recall**, **F1-score** y **ROC-AUC**.

### 2.3 Variables numéricas y correlaciones

In [ ]:
num_cols_eda = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
corr = eda_df[num_cols_eda].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f")
plt.title("Matriz de correlación - variables numéricas")
plt.show()

corr

### 2.4 Relación entre variables importantes y Churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=eda_df, x="Contract", hue="Churn", ax=axes[0, 0])
axes[0, 0].set_title("Churn por tipo de contrato")
axes[0, 0].tick_params(axis='x', rotation=15)

sns.histplot(data=eda_df, x="tenure", hue="Churn", multiple="stack", bins=30, ax=axes[0, 1])
axes[0, 1].set_title("Distribución de tenure por churn")

sns.boxplot(data=eda_df, x="Churn", y="MonthlyCharges", ax=axes[1, 0])
axes[1, 0].set_title("MonthlyCharges por churn")

sns.boxplot(data=eda_df, x="Churn", y="TotalCharges", ax=axes[1, 1])
axes[1, 1].set_title("TotalCharges por churn")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=eda_df, x="InternetService", hue="Churn", ax=axes[0])
axes[0].set_title("Churn por InternetService")

sns.countplot(data=eda_df, x="PaymentMethod", hue="Churn", ax=axes[1])
axes[1].set_title("Churn por PaymentMethod")
axes[1].tick_params(axis='x', rotation=30)

sns.countplot(data=eda_df, x="PaperlessBilling", hue="Churn", ax=axes[2])
axes[2].set_title("Churn por PaperlessBilling")

plt.tight_layout()
plt.show()

### 2.5 Sesgos adicionales a comentar en el informe
Puedes incluir observaciones como estas:

- **Desbalance de clases**: normalmente hay más clientes que no cancelan que clientes que sí cancelan.  
- **Posible sesgo por antigüedad** (`tenure`): clientes con menor permanencia suelen mostrar mayor churn.  
- **Posible sesgo por contrato**: contratos *month-to-month* suelen asociarse a mayor abandono.  
- **Posible sesgo por forma de pago**: ciertos métodos de pago pueden concentrar más churn.  

> Nota: aquí "sesgo" se interpreta como **desbalance o concentración no uniforme** en ciertas subpoblaciones, no necesariamente como sesgo algorítmico ético.

### Conclusión del EDA
El análisis exploratorio permite ver que:
- `TotalCharges` requiere tratamiento por valores faltantes implícitos.
- Existe **desbalance de clases**, por lo que se necesitan métricas más allá de accuracy.
- Variables como `Contract`, `tenure`, `MonthlyCharges` e `InternetService` parecen tener relación con la fuga de clientes.

## 3. Preprocesamiento

Se implementará un pipeline reproducible para evitar **data leakage**, incluyendo:

- limpieza de `TotalCharges`,
- eliminación de identificadores no predictivos,
- codificación de la variable objetivo,
- imputación,
- codificación One-Hot para variables categóricas,
- escalamiento para variables numéricas.

In [ ]:
data = df.copy()

# Limpieza básica
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Eliminar identificador único
data = data.drop(columns=["customerID"])

# Variable objetivo
y = data["Churn"].copy()
X = data.drop(columns=["Churn"]).copy()

# Codificación de la etiqueta objetivo (Label Encoding)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)  # No -> 0, Yes -> 1

print("Clases objetivo:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
X.head()

In [ ]:
# Identificación de columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numéricas:", numeric_features)
print("\nVariables categóricas:", categorical_features)

In [ ]:
# Split estratificado para conservar proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print("Tamaño train:", X_train.shape)
print("Tamaño test:", X_test.shape)

### 3.1 Transformaciones
- Numéricas: imputación con mediana + `StandardScaler`.
- Categóricas: imputación con moda + `OneHotEncoder`.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### Conclusión del preprocesamiento
La limpieza y transformación quedan integradas dentro de un `Pipeline`, lo cual asegura que la imputación, el escalamiento y la codificación se ajusten **solo con los datos de entrenamiento** durante la validación cruzada.

## 4. Ingeniería de Características

Se aplicará **selección de variables** mediante un método estadístico:
- `SelectKBest` con `mutual_info_classif`

Esto permite conservar las características más informativas luego del preprocesamiento.

> Ventaja: la selección ocurre dentro del pipeline, evitando fuga de información.

In [ ]:
# Selector de variables
feature_selector = SelectKBest(score_func=mutual_info_classif, k=20)
feature_selector

### Nota metodológica
El número `k=20` será tratado como hiperparámetro dentro de la búsqueda para permitir que cada modelo determine cuántas variables procesadas le convienen más.

## 5. Entrenamiento y Ajuste de Modelos

Se entrenarán y ajustarán tres modelos:

1. **Logistic Regression**
2. **KNN**
3. **Random Forest**

La búsqueda de hiperparámetros se hará con **GridSearchCV** y validación cruzada estratificada.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [ ]:
pipelines = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),

    "KNN": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", KNeighborsClassifier())
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE))
    ])
}

param_grids = {
    "Logistic Regression": {
        "selector__k": [10, 20, "all"],
        "model__C": [0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
        "model__solver": ["liblinear"]
    },

    "KNN": {
        "selector__k": [10, 20, "all"],
        "model__n_neighbors": [5, 11, 21],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["minkowski"]
    },

    "Random Forest": {
        "selector__k": [10, 20, "all"],
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 15],
        "model__min_samples_split": [2, 10],
        "model__class_weight": [None, "balanced"]
    }
}

In [ ]:
best_models = {}
cv_summary = []

for model_name in pipelines:
    print("=" * 80)
    print(f"Entrenando: {model_name}")

    grid = GridSearchCV(
        estimator=pipelines[model_name],
        param_grid=param_grids[model_name],
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0
    )

    grid.fit(X_train, y_train)
    best_models[model_name] = grid.best_estimator_

    # CV robusta del mejor modelo (múltiples métricas)
    cv_results = cross_validate(
        grid.best_estimator_,
        X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    row = {
        "Modelo": model_name,
        "Mejores hiperparámetros": grid.best_params_,
        "CV Accuracy": np.mean(cv_results["test_accuracy"]),
        "CV Precision": np.mean(cv_results["test_precision"]),
        "CV Recall": np.mean(cv_results["test_recall"]),
        "CV F1": np.mean(cv_results["test_f1"]),
        "CV ROC-AUC": np.mean(cv_results["test_roc_auc"]),
    }
    cv_summary.append(row)

cv_results_df = pd.DataFrame(cv_summary)
cv_results_df.sort_values(by="CV F1", ascending=False)

### Conclusión del entrenamiento
La comparación entre modelos se hará con base en **F1-score** y **ROC-AUC**, ya que el problema presenta desbalance de clases y no conviene depender únicamente de `accuracy`.

## 6. Validación Robusta: métricas en Test set

Ahora se evalúa cada mejor modelo en el conjunto de prueba independiente.

In [ ]:
test_summary = []

for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    row = {
        "Modelo": model_name,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    }
    test_summary.append(row)

test_results_df = pd.DataFrame(test_summary).sort_values(by="Test F1", ascending=False)
test_results_df

In [ ]:
# Modelo ganador por F1 en test
best_model_name = test_results_df.iloc[0]["Modelo"]
best_model = best_models[best_model_name]

print("Mejor modelo en test (según F1):", best_model_name)

In [ ]:
# Predicciones del mejor modelo
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_best, target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f"Matriz de confusión - {best_model_name}")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob_best)
plt.title(f"Curva ROC - {best_model_name}")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_prob_best)
plt.title(f"Curva Precision-Recall - {best_model_name}")
plt.show()

## 7. Importancia de variables / variables seleccionadas

Como el taller pide selección de variables basada en importancia o métodos estadísticos, se documentan las variables retenidas por `SelectKBest`.

In [ ]:
# Obtener nombres de variables transformadas por el preprocesador
fitted_preprocessor = best_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()

selector = best_model.named_steps["selector"]
support_mask = selector.get_support()

selected_features = pd.DataFrame({
    "feature": feature_names,
    "selected": support_mask
})

selected_features = selected_features[selected_features["selected"]].reset_index(drop=True)
selected_features.head(20)

In [ ]:
print(f"Número de variables seleccionadas: {selected_features.shape[0]}")
selected_features

In [ ]:
# Si el mejor modelo es Random Forest, mostrar importancia de variables seleccionadas
if best_model_name == "Random Forest":
    importances = best_model.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "feature": selected_features["feature"],
        "importance": importances
    }).sort_values("importance", ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x="importance", y="feature")
    plt.title("Top 15 variables más importantes - Random Forest")
    plt.show()
else:
    # Para Logistic Regression, usar magnitud absoluta de coeficientes
    if best_model_name == "Logistic Regression":
        coefs = np.abs(best_model.named_steps["model"].coef_[0])
        coef_df = pd.DataFrame({
            "feature": selected_features["feature"],
            "importance": coefs
        }).sort_values("importance", ascending=False).head(15)

        plt.figure(figsize=(10, 6))
        sns.barplot(data=coef_df, x="importance", y="feature")
        plt.title("Top 15 variables más influyentes - Logistic Regression")
        plt.show()
    else:
        display(Markdown("El modelo ganador no expone una importancia directa tan interpretable como Random Forest o Logistic Regression."))

## 8. Conclusiones finales

### Hallazgos principales
- El dataset presenta **desbalance de clases**, por lo que fue necesario evaluar con **F1-score** y **ROC-AUC**.
- `TotalCharges` requirió limpieza porque contiene valores faltantes implícitos al convertir a tipo numérico.
- Variables como `Contract`, `tenure`, `MonthlyCharges` y servicios asociados mostraron relación con la fuga de clientes.

### Sobre el pipeline
- Se implementó un flujo completo con **imputación**, **One-Hot Encoding**, **escalamiento** y **selección de variables**.
- Todo se integró en un `Pipeline`, lo que evita **data leakage** durante la validación cruzada.

### Sobre los modelos
- Se compararon **3 modelos**: Logistic Regression, KNN y Random Forest.
- El modelo final se seleccionó con base en desempeño en **Cross-Validation** y su capacidad de generalizar en el **Test set**.

### Recomendación para el informe
En el reporte final puedes argumentar que el mejor modelo fue seleccionado no solo por accuracy, sino por su equilibrio entre:
- **precision**,
- **recall**,
- **F1-score**,
- **ROC-AUC**,

lo cual es especialmente importante en problemas con clases desbalanceadas como churn.

## 9. Opcional: guardar el mejor modelo

Si luego quieres usar este notebook para el dashboard de Streamlit, puedes guardar el mejor pipeline así:

In [ ]:
# import joblib
# joblib.dump(best_model, "best_telco_churn_model.pkl")